In [1]:
# ============================================
# County Boundaries Dataset
# K-Means Clustering + NLP
# ============================================

# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


# ============================================
# 2. Load the dataset
# ============================================

file_path = "county-boundaries.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================
# 3. Basic data cleaning
# ============================================

print("\nMissing values:")
print(df.isnull().sum())

# Remove completely empty rows
df = df.dropna(how="all").reset_index(drop=True)

print("\nDataset after cleaning:", df.shape)


# ============================================
# 4. Find a suitable text column
# ============================================

possible_text_columns = [
    "county",
    "County",
    "name",
    "Name",
    "description",
    "Description",
    "text",
    "Text"
]

text_column = None

for column in possible_text_columns:
    if column in df.columns:
        text_column = column
        break

# If no obvious text column exists, find the first object/string column
if text_column is None:
    text_columns = df.select_dtypes(include=["object"]).columns

    if len(text_columns) > 0:
        text_column = text_columns[0]

if text_column is None:
    raise ValueError(
        "No text column was found. Please specify the county/name/description column manually."
    )

print("Text column selected:", text_column)


# ============================================
# 5. Clean the text
# ============================================

df[text_column] = (
    df[text_column]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

# Remove rows with empty text
df = df[df[text_column] != ""].reset_index(drop=True)

print("Rows available for NLP:", len(df))


# ============================================
# 6. NLP - TF-IDF
# ============================================

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=1000,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df[text_column])

print("TF-IDF matrix shape:", X.shape)


# ============================================
# 7. Find a suitable number of clusters
# ============================================

# Test different numbers of clusters
cluster_range = range(2, 11)

inertias = []
silhouette_scores = []

for k in cluster_range:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X)

    inertias.append(kmeans.inertia_)

    silhouette_scores.append(
        silhouette_score(X, labels)
    )


# ============================================
# 8. Elbow Method
# ============================================

plt.figure(figsize=(8, 5))

plt.plot(
    list(cluster_range),
    inertias,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")

plt.show()


# ============================================
# 9. Silhouette Score
# ============================================

plt.figure(figsize=(8, 5))

plt.plot(
    list(cluster_range),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")

plt.show()


# ============================================
# 10. Apply K-Means
# ============================================

# Change this value after examining the graphs
K = 5

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)


# ============================================
# 11. Display clustered counties
# ============================================

print("Cluster counts:")

print(
    df["cluster"]
    .value_counts()
    .sort_index()
)

display(
    df[[text_column, "cluster"]]
    .sort_values("cluster")
)


# ============================================
# 12. Extract important words for each cluster
# ============================================

terms = vectorizer.get_feature_names_out()

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

print("\nImportant terms in each cluster:\n")

for cluster_number in range(K):

    top_terms = [
        terms[index]
        for index in order_centroids[cluster_number, :10]
    ]

    print(
        f"Cluster {cluster_number}: "
        + ", ".join(top_terms)
    )


# ============================================
# 13. Reduce dimensions using PCA
# ============================================

# Convert sparse TF-IDF matrix to dense format
X_dense = X.toarray()

pca = PCA(n_components=2, random_state=42)

X_pca = pca.fit_transform(X_dense)

df["PCA_1"] = X_pca[:, 0]
df["PCA_2"] = X_pca[:, 1]


# ============================================
# 14. Visualise the clusters
# ============================================

plt.figure(figsize=(10, 7))

for cluster_number in range(K):

    cluster_data = df[df["cluster"] == cluster_number]

    plt.scatter(
        cluster_data["PCA_1"],
        cluster_data["PCA_2"],
        label=f"Cluster {cluster_number}",
        alpha=0.7
    )

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")

plt.title("County Clusters using K-Means + NLP")

plt.legend()

plt.show()


# ============================================
# 15. Save the results
# ============================================

output_file = "county-boundaries-clustered.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Clustered dataset saved as: {output_file}")

Dataset shape: (42972, 3)

Columns:
['x', 'y', 'id']

First 5 rows:


,x,y,id
0,-86.413116,32.707386,1.001
1,-86.411272,32.421436,1.001
2,-86.411172,32.409937,1.001
3,-86.425072,32.402437,1.001
4,-86.798268,32.308632,1.001



Missing values:
x     0
y     0
id    0
dtype: int64

Dataset after cleaning: (42972, 3)


ValueError: No text column was found. Please specify the county/name/description column manually.